# Task 4: General Health Query Chatbot (Prompt Engineering)

**Internship:** DevelopersHub Corporation – AI/ML Engineering  
**Objective:** Create a chatbot that answers general health-related questions using an LLM via Groq API.  
**Tool:** Groq API (Free) — openai/gpt-oss-120b model  
**Technique:** Prompt Engineering + Safety Filtering

## Problem Statement

Build a chatbot that can answer general health-related questions using a Large Language Model (LLM). The chatbot uses prompt engineering to give friendly, clear, and safe health information while filtering out harmful queries.

## Goal

- Connect to Groq API and use a free LLM model.
- Use prompt engineering to define chatbot behavior.
- Add safety filters to block harmful queries.
- Build a conversational loop for user interaction.
- Test with example health queries.

## Importing Required Libraries

In [ ]:
from groq import Groq

## Setup Groq Client

Get your free API key from: https://console.groq.com/keys  
Replace `your-groq-api-key-here` with your actual key.

In [ ]:
# Add your Groq API key here
client = Groq(api_key="your-groq-api-key-here")

## System Prompt (Prompt Engineering)

This is the core of prompt engineering — we define the chatbot's role, tone, and behavior using a system prompt.

In [ ]:
SYSTEM_PROMPT = """
You are a friendly and knowledgeable health assistant.
Your job is to answer general health-related questions in a
clear, simple, and easy-to-understand way.

Follow these rules strictly:
1. Give helpful and accurate general health information.
2. Always be warm, friendly, and supportive in tone.
3. NEVER diagnose a disease or prescribe medication.
4. NEVER give specific dosage instructions for any medicine.
5. For any serious symptoms, always recommend seeing a doctor.
6. Keep your answers concise — 3 to 5 sentences maximum.
7. End every response with: 'Please consult a doctor for
   personalized medical advice.'
"""

print("System prompt defined successfully.")

## Safety Filter

Block harmful or dangerous queries before sending them to the API.

In [ ]:
UNSAFE_KEYWORDS = [
    "suicide", "kill myself", "overdose", "how to die",
    "self harm", "cut myself", "end my life", "poison",
    "drug abuse", "illegal drugs"
]

def is_unsafe(user_input):
    """Check if user query contains unsafe content."""
    user_input_lower = user_input.lower()
    for keyword in UNSAFE_KEYWORDS:
        if keyword in user_input_lower:
            return True
    return False

# Test safety filter
print("Safety filter test:")
print("'What causes fever?' is unsafe:", is_unsafe("What causes fever?"))
print("'suicide' is unsafe:", is_unsafe("suicide"))

## Get Health Response Function

Send user query to Groq API and return response. Auto tries multiple models.

In [ ]:
# Models available in Groq account
MODELS = [
    "openai/gpt-oss-120b",
    "openai/gpt-oss-20b",
    "groq/compound",
    "groq/compound-mini",
    "openai/gpt-oss-safeguard-20b",
]

def get_health_response(user_query):
    """Send user query to Groq API and return response."""
    for model in MODELS:
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": user_query}
                ],
                temperature=0.7,
                max_tokens=300
            )
            return response.choices[0].message.content

        except Exception as e:
            error_msg = str(e)
            if any(x in error_msg for x in
                   ["model_not_found", "decommissioned",
                    "404", "does not exist", "no access"]):
                continue
            elif any(x in error_msg for x in
                     ["authentication", "api_key", "401", "invalid_api_key"]):
                return "ERROR: Invalid API key. Check https://console.groq.com/keys"
            elif any(x in error_msg for x in ["rate_limit", "429"]):
                return "ERROR: Rate limit reached. Please wait and try again."
            else:
                return f"ERROR: {error_msg}"

    return "ERROR: No models available."

print("Function defined successfully.")

## Test With Example Queries

Testing the chatbot with the example queries mentioned in the task requirements.

In [ ]:
# Test Query 1 — from task requirements
query1 = "What causes a sore throat?"
print(f"Query: {query1}")
print(f"Response: {get_health_response(query1)}")

In [ ]:
# Test Query 2 — from task requirements
query2 = "Is paracetamol safe for children?"
print(f"Query: {query2}")
print(f"Response: {get_health_response(query2)}")

In [ ]:
# Test Query 3
query3 = "How can I lower my blood pressure naturally?"
print(f"Query: {query3}")
print(f"Response: {get_health_response(query3)}")

In [ ]:
# Test Safety Filter
unsafe_query = "how to overdose"
print(f"Query: {unsafe_query}")
if is_unsafe(unsafe_query):
    print("Response: I'm sorry, I'm not able to help with that topic.")
    print("          If you are in distress, please contact a mental")
    print("          health professional or call a helpline.")

## Run Interactive Chatbot

Run the full chatbot loop — type your questions and get responses.

In [ ]:
def run_chatbot():
    """Main chatbot loop."""
    print("\n" + "=" * 50)
    print("   GENERAL HEALTH QUERY CHATBOT")
    print("   DevelopersHub – AI/ML Internship Task 4")
    print("   Powered by Groq API (Free)")
    print("=" * 50)
    print("Ask any general health question.")
    print("Type 'quit' to exit.\n")

    print("Example questions you can ask:")
    print("  → What causes a sore throat?")
    print("  → Is paracetamol safe for children?")
    print("  → How can I lower my blood pressure naturally?")
    print("  → What are the symptoms of diabetes?")
    print("  → How many hours of sleep does an adult need?")
    print("-" * 50 + "\n")

    while True:
        user_input = input("You: ").strip()

        if user_input.lower() in ['quit', 'exit', 'bye']:
            print("\nChatbot: Thank you! Stay healthy!")
            break

        if not user_input:
            print("Chatbot: Please type a question.\n")
            continue

        if is_unsafe(user_input):
            print("\nChatbot: I'm sorry, I'm not able to help with that topic.")
            print("         Please contact a mental health professional.\n")
            continue

        print("\nChatbot: ", end="", flush=True)
        response = get_health_response(user_input)
        print(response)
        print()

run_chatbot()

# Final Insights

### Observations

1. Prompt engineering successfully controls the chatbot's tone and behavior.
2. The system prompt ensures responses are always friendly and safe.
3. Safety filter correctly blocks harmful queries before reaching the API.
4. Groq API provides free, fast LLM responses with no cost.
5. Auto model fallback ensures chatbot works even if one model is unavailable.

### Conclusion

The Health Query Chatbot successfully answers general health questions using prompt engineering with the Groq API. The system prompt defines clear boundaries — the chatbot never diagnoses or prescribes, always recommends doctor consultation, and blocks unsafe queries automatically.